Running Ironhack notebooks in VS Code

One-off setup (already done)
Your .venv_airflow environment already has ipykernel and pandas installed, so you don't need to repeat this. It's here in case you ever make a new environment or need a new package.

1. In the VS Code terminal, make sure you're in ~/data_course.
2. Switch the environment on: source .venv_airflow/bin/activate (the prompt should start with (.venv_airflow)).
3. Install what notebooks need: pip install ipykernel pandas
4. For any other package a lab needs later (for example numpy), use the same pattern: pip install <package name>

Every time you add a new notebook

Each new .ipynb file needs its kernel picking once. After that, VS Code remembers it for that file.

1. Drag it in. Drag the downloaded .ipynb from Windows into your 01_notebooks folder in the VS Code Explorer (left-hand panel). Rename it if you like (right-click → Rename), for example replacing DATE with the date.
2. Open it. Click the file in the Explorer.
3. Click Select Kernel. It's in the top-right corner of the notebook.
4. Click Python Environments…
5. Pick .venv_airflow. Choose the line ending .venv_airflow/bin/python. Ignore the one marked "Recommended" (/usr/bin/python3): that's Ubuntu's own Python and it won't work.
6. Check the top-right. It should now say .venv_airflow.
7. Run the import cell. Click ▷ next to import pandas as pd. A green tick and [1] mean it's working.

In [1]:
import pandas as pd

## Part 1 — Source Data and Validation

### Q. 1 — Validate a Customer Batch

Use:

```python
import pandas as pd

customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "customer_name": [
        "Amelia Clarke",
        "Oliver Bennett",
        "Aisha Khan",
        "Harry Thompson",
    ],
    "email": [
        "amelia@example.com",
        "",
        "aisha@example.com",
        None,
    ],
})
```

Create:

```python
valid_customers
invalid_customers
```

Rule:

```python
customer_id and customer_name are required
email can be blank
```

In [ ]:
customers = pd.DataFrame({"customer_id": [1, 2, 3, 4],"customer_name": ["Amelia Clarke", "Oliver Bennett", "Aisha Khan", "Harry Thompson"], "email": ["amelia@example.com", "", "aisha@example.com", None]})

invalid_mask = (customers["customer_name"].isna() | customers["email"].isna())

invalid_customers = customers[invalid_mask].copy()
valid_customers = customers[~invalid_mask].copy()

bronze_customers = (len(customers))
invalid_customers = (len(invalid_customers))
valid_customers = (len(valid_customers))
difference = bronze_customers - (invalid_customers + valid_customers)
print(f"There are {bronze_customers} rows in the bronze data. The validated data shows {invalid_customers} invalid rows and {valid_customers} valid rows. The total missing rows from validation is {difference}.")
                                                                                                                                                      

There are 4 rows in the bronze data. The validated data shows 1 invalid rows and 3 valid rows. The total missing rows from validation is 0.


### Q. 2 — Validate an Order Batch

Use:

```python
import pandas as pd

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer_id": [1, 2, 3, 4, 1],
    "quantity": [2, 1, 0, 3, 1],
    "unit_price_gbp": [25.00, 40.00, 12.00, None, 50.00],
    "status": [
        "complete",
        "complete",
        "complete",
        "complete",
        "cancelled",
    ],
})
```

Reject rows when:

- quantity is 0 or negative;
- price is missing.

Create valid and rejected DataFrames.

In [19]:
orders = pd.DataFrame({"order_id": [101, 102, 103, 104, 105], "customer_id": [1, 2, 3, 4, 1], "quantity": [2, 1, 0, 3, 1], "unit_price_gbp": [25.00, 40.00, 12.00, None, 50.00], "status": ["complete", "complete", "complete", "complete", "cancelled"]})

invalid_orders_mask = (orders["unit_price_gbp"].isna() | orders["quantity"] <= 0)

invalid_orders = orders[invalid_orders_mask].copy()
valid_orders = orders[~invalid_orders_mask].copy()

bronze_orders = (len(orders))
invalid_orders = (len(invalid_orders))
valid_orders = (len(valid_orders))
difference = bronze_orders - (invalid_orders + valid_orders)
print(f"There are {bronze_orders} rows in the bronze data. The validated data shows {invalid_orders} invalid rows and {valid_orders} valid rows. The total missing rows from validation is {difference}.")
                                                                                                                                                      

There are 5 rows in the bronze data. The validated data shows 1 invalid rows and 4 valid rows. The total missing rows from validation is 0.


### Q. 3 — Validate Three Daily Batches

Use:

```python
import pandas as pd

batch_1 = pd.DataFrame({
    "order_id": [201, 202],
    "amount_gbp": [45.50, 30.00],
})

batch_2 = pd.DataFrame({
    "order_id": [203, 204],
    "amount_gbp": [0, 62.25],
})

batch_3 = pd.DataFrame({
    "order_id": [205, 206],
    "amount_gbp": [-5.00, 20.00],
})
```

For each batch, print:

```python
source_count
valid_count
rejected_count
```

A valid amount must be greater than 0.

In [ ]:
batch_1 = pd.DataFrame({"order_id": [201, 202], "amount_gbp": [45.50, 30.00]})
batch_2 = pd.DataFrame({"order_id": [203, 204], "amount_gbp": [0, 62.25]}) 
batch_3 = pd.DataFrame({"order_id": [205, 206], "amount_gbp": [-5.00, 20.00]})

# Can't use straight forward pd.concat because once the batches are stacked together you lose track of which row came from which batch. But there is a way around this in pandas. I'm going to use standard pyhton though. 

batches = [batch_1, batch_2, batch_3]

source_count = []
valid_count = []
rejected_count = []

for batch in batches:
    source_count.append(len(batch))
    valid = batch[batch["amount_gbp"] > 0]
    rejected = batch[batch["amount_gbp"] <= 0]
    valid_count.append(len(valid))
    rejected_count.append(len(rejected))

print(source_count)
print(valid_count)
print(rejected_count)

# Ironhacks answer. My answer doesn't really answer the question:

batches = {"batch_1": batch_1, "batch_2": batch_2, "batch_3": batch_3}
 
for batch_name, batch in batches.items(): 
# '.items() hands you the dictionary’s entries one at a time as pairs: first ("batch_1", batch_1), then ("batch_2", batch_2), then ("batch_3", batch_3). The for batch_name, batch in ... part then splits each pair into two names as it goes. batch_name gets the label, such as "batch_1", which is just text, and batch gets the DataFrame itself, so you can filter it.'
# 'batch_name, batch...Those are just names you make up. Python doesn’t know or care what they mean. You could write: for x, y in batches.items(): valid = y[y["amount_gbp"] > 0] print(x, len(valid))'
    valid = batch[batch["amount_gbp"] > 0]
    rejected = batch[batch["amount_gbp"] <= 0]
 
    print(batch_name, "source =", len(batch), "valid =", len(valid), "rejected =", len(rejected))

[2, 2, 2]
[2, 1, 1]
[0, 1, 1]
batch_1 source = 2 valid = 2 rejected = 0
batch_2 source = 2 valid = 1 rejected = 1
batch_3 source = 2 valid = 1 rejected = 1


### Q. 4 — MCQ: Validation

Why should row counts be checked during validation?

A. To confirm that rows were not silently lost or unexpectedly added

B. To prove every business value is correct automatically

C. To remove the need for business rules

D. To make all files the same size

========================================================================
ANSWER Q4
========================================================================

A. To confirm that rows were not silently lost or unexpectedly added.

========================================================================

## Part 2 — Rejecting, Fixing, and Safe Joins

### Q. 5 — Reject or Fix?

For each issue, choose fix, reject, or review.

```python
A. City contains extra spaces: " London "
B. Unit price is missing.
C. Email is blank but email is optional.
D. Quantity is -3.
E. City is "london" but the required standard is "London".
```

========================================================================
ANSWER Q5
========================================================================

A. City contains extra spaces: " London "

Fix.

====================================

B. Unit price is missing.

Revue then likely reject. 

====================================

C. Email is blank but email is optional.

N/A.

====================================

D. Quantity is -3.

Reject.

====================================

E. City is "london" but the required standard is "London".

Fix.

========================================================================

### Q. 6 — Fix Safe Values and Reject Unsafe Values

Use:

```python
import pandas as pd

orders = pd.DataFrame({
    "order_id": [301, 302, 303, 304],
    "city": [" London ", "leeds", "Bristol", "Manchester"],
    "quantity": [2, 1, -1, 3],
    "unit_price_gbp": [25.00, 20.00, 15.00, None],
})
```

Rules:

- trim spaces;
- convert city to title case;
- reject non-positive quantity;
- reject missing price.

In [ ]:
orders = pd.DataFrame({"order_id": [301, 302, 303, 304], "city": [" London ", "leeds", "Bristol", "Manchester"], "quantity": [2, 1, -1, 3], "unit_price_gbp": [25.00, 20.00, 15.00, None],})

silver_1 = orders.copy()

def trim_text_columns(orders_df): # This is new. I used this to go through the whole df that was string, not just the column in question.
    for column in orders_df.columns:
        if pd.api.types.is_string_dtype(orders_df[column]):
            orders_df[column] = orders_df[column].str.strip()
    return orders_df
silver_1 = trim_text_columns(silver_1)

silver_1["city"] = silver_1["city"].str.title()

invalid_mask = (silver_1["quantity"] <= 0) | (silver_1["unit_price_gbp"].isna())

rejected_orders = silver_1[invalid_mask].copy()
valid_orders = silver_1[~invalid_mask].copy()

print(valid_orders)
print(rejected_orders)

   order_id    city  quantity  unit_price_gbp
0       301  London         2            25.0
1       302   Leeds         1            20.0
   order_id        city  quantity  unit_price_gbp
2       303     Bristol        -1            15.0
3       304  Manchester         3             NaN


### Q. 7 — Perform a Safe Many-to-One Join

Use:

```python
import pandas as pd

customers = pd.DataFrame({
    "customer_id": [1, 2, 3],
    "customer_name": [
        "Amelia Clarke",
        "Oliver Bennett",
        "Aisha Khan",
    ],
    "city": [
        "London",
        "Manchester",
        "Leeds",
    ],
})

orders = pd.DataFrame({
    "order_id": [401, 402, 403, 404],
    "customer_id": [1, 2, 1, 3],
    "amount_gbp": [45.50, 30.00, 62.25, 20.00],
})
```

Join orders to customers.

Use Pandas join validation to confirm that many orders can match one customer.

In [33]:
customers = pd.DataFrame({"customer_id": [1, 2, 3], "customer_name": ["Amelia Clarke", "Oliver Bennett", "Aisha Khan"], "city": ["London", "Manchester", "Leeds"]})
orders = pd.DataFrame({"order_id": [401, 402, 403, 404], "customer_id": [1, 2, 1, 3], "amount_gbp": [45.50, 30.00, 62.25, 20.00]})

merged = orders.merge(customers, on = "customer_id", how = "left", validate = "many_to_one") # The many to one bit is new. 
print(merged)



   order_id  customer_id  amount_gbp   customer_name        city
0       401            1       45.50   Amelia Clarke      London
1       402            2       30.00  Oliver Bennett  Manchester
2       403            1       62.25   Amelia Clarke      London
3       404            3       20.00      Aisha Khan       Leeds


### Q. 8 — MCQ: Safe Join

Why validate a many-to-one join?

A. To make the join result sorted

B. To remove every unmatched row automatically

C. To convert all columns to text

D. To check that the lookup side does not unexpectedly contain duplicate keys

========================================================================
ANSWER Q8
========================================================================

D. To check that the lookup side does not unexpectedly contain duplicate keys.

"""
The answer is D: to check that the lookup side doesn't unexpectedly contain duplicate keys. In a many-to-one join, the "many" side is orders, since one customer can have lots of orders. The "one" side is customers, where each `customer_id` should appear only once. If a customer ID is accidentally duplicated, every order for that customer gets matched twice, so your order count and revenue quietly double. Using `validate="many_to_one"` makes pandas stop with an error instead of letting that happen. That's exactly what your mock data tests. Customer 6 appears twice in `customers.csv`, and order 811 belongs to customer 6. So if you join before removing the duplicate, the validation should fail, which is what Lab 1 Q18 demonstrates. The other options are wrong because validation doesn't sort anything (A), unmatched rows depend on the join type, not validation (B), and it doesn't change column types (C).
"""

========================================================================

## Part 3 — Derived Columns, Curated Output, and Raw Preservation

### Q. 9 — Create a Derived Revenue Column

Use:

```python
import pandas as pd

orders = pd.DataFrame({
    "order_id": [501, 502, 503],
    "quantity": [2, 1, 4],
    "unit_price_gbp": [25.00, 40.00, 12.50],
})
```

Create:

```python
line_revenue_gbp
```

as:

```python
quantity × unit_price_gbp
```

In [34]:
orders = pd.DataFrame({"order_id": [501, 502, 503], "quantity": [2, 1, 4], "unit_price_gbp": [25.00, 40.00, 12.50]})

orders["live_revenue_gbp"] = orders["quantity"] * orders["unit_price_gbp"]

print(orders)

   order_id  quantity  unit_price_gbp  live_revenue_gbp
0       501         2            25.0              50.0
1       502         1            40.0              40.0
2       503         4            12.5              50.0


### Q. 10 — Build a Curated Daily Sales Report

Use:

```python
import pandas as pd

orders = pd.DataFrame({
    "order_date": [
        "2026-08-01",
        "2026-08-01",
        "2026-08-01",
        "2026-08-02",
    ],
    "city": [
        "London",
        "London",
        "Manchester",
        "Leeds",
    ],
    "amount_gbp": [
        45.50,
        62.25,
        30.00,
        20.00,
    ],
    "status": [
        "complete",
        "complete",
        "cancelled",
        "complete",
    ],
})
```

Create a report with:

```python
order_date
city
completed_revenue_gbp
```

In [37]:
orders = pd.DataFrame({"order_date": ["2026-08-01", "2026-08-01", "2026-08-01", "2026-08-02"], "city": ["London", "London", "Manchester", "Leeds"], "amount_gbp": [45.50, 62.25, 30.00, 20.00], "status": ["complete", "complete", "cancelled", "complete"]})

gold = orders.copy()

gold = gold[gold["status"] == "complete"]

gold_report = gold.groupby(["order_date", "city"], as_index = False).agg(completed_revenue_gbp=("amount_gbp", "sum")).sort_values("completed_revenue_gbp", ascending = False)
print(gold_report)

   order_date    city  completed_revenue_gbp
0  2026-08-01  London                 107.75
1  2026-08-02   Leeds                  20.00


### Q. 11 — Preserve Raw Data

Use:

```python
import pandas as pd

source = pd.DataFrame({
    "order_id": [601, 602, 603],
    "city": [" London ", "Leeds", "Manchester"],
})
```

Create:

```python
raw_orders
clean_orders
```

Trim spaces only in `clean_orders`.

Prove that `raw_orders` still contains the original `" London "` value.

In [ ]:
source = pd.DataFrame({"order_id": [601, 602, 603], "city": [" London ", "Leeds", "Manchester"]})

bronze = source.copy()

def trim_text_columns_2(source_df): 
    for column in source_df.columns:
        if pd.api.types.is_string_dtype(source_df[column]):
            source_df[column] = source_df[column].str.strip()
    return source_df
silver_1 = trim_text_columns_2(source)

bronze_source = (len(bronze))
silver_source = (len(silver_1))
difference = bronze_source - silver_source
print(f"There are {bronze_source} rows in the bronze data. The silver data shows {silver_source} rows. The total missing rows from validation is {difference}.")

print("Original raw value:", repr(bronze.loc[0, "city"])) # New: repr() shows a value exactly as Python stores it, including the quote marks around text. That makes spaces at the start and end visible.

There are 3 rows in the bronze data. The silver data shows 3 rows. The total missing rows from validation is 0.
Original raw value: ' London '


### Q. 12 — MCQ: Derived Column

Which is a derived column?

A. `customer_id` copied directly from the source

B. `line_revenue_gbp = quantity × unit_price_gbp`

C. `order_id` received from the source

D. `city` received from the source

========================================================================
ANSWER Q12
========================================================================

 B. line_revenue_gbp = quantity × unit_price_gbp.

========================================================================

## Part 4 — Audit Records and Data-Quality Tests

### Q. 13 — Create an Audit Record

A pipeline run has:

```python
run_id = run_20260801
source_count = 100
valid_count = 94
rejected_count = 6
status = success
```

Create a Python dictionary containing these values and check:

```python
source_count == valid_count + rejected_count
```

In [44]:
q13 = {"run_id": ["run_20260801"], "source_count": [100], "valid_count": [94], "rejected_count": [6], "status": ["success"]}

source_count = q13["source_count"][0] # Add [0] when you pull each value out. That takes the first (and only) item from the list, so you get the number itself:
valid_count = q13["valid_count"][0]
rejected_count = q13["rejected_count"][0]

difference = source_count - (valid_count + rejected_count)

print(f"There are {source_count} rows in the source. The validated data shows {valid_count} valid rows and {rejected_count} rejected rows. The total missing rows from validation is {difference}.")


There are 100 rows in the source. The validated data shows 94 valid rows and 6 rejected rows. The total missing rows from validation is 0.


### Q. 14 — Run Data-Quality Tests

Use:

```python
import pandas as pd

orders = pd.DataFrame({
    "order_id": [701, 702, 702, 704],
    "customer_id": [1, 2, 2, None],
    "amount_gbp": [45.50, 30.00, 30.00, -5.00],
})
```

Calculate:

```python
duplicate order IDs
missing customer IDs
non-positive amounts
```

In [ ]:
orders = pd.DataFrame({"order_id": [701, 702, 702, 704], "customer_id": [1, 2, 2, None], "amount_gbp": [45.50, 30.00, 30.00, -5.00]})

duplicate_mask = orders.duplicated(subset=["order_id"], keep = "first")
duplicates = orders[duplicate_mask].copy()
print(f"The duplicate orders are order_id's: {duplicates["order_id"]}.")
# Why does this work, I thought it would need to be: 'duplicates = orders[~duplicate_mask].copy()'. 'It’s the other way round. duplicated() marks the repeats as True, so filtering with the mask as-is gives you the duplicates.'

#I know how to do this, it is just repetitive, so will copy and paste the Ironhack answer for the remaining two. They use a different format to the above, prefer my format. 
missing_customer_count = (orders["customer_id"].isna().sum())
print("Missing customer IDs:", missing_customer_count)

invalid_amount_count = (orders["amount_gbp"] <= 0).sum()
print("Non-positive amounts:",invalid_amount_count)



The duplicate orders are order_id's: 2    702
Name: order_id, dtype: int64.
Missing customer IDs: 1
Non-positive amounts: 1


### Q. 15 — Reconcile Three Batch Audits

Use:

```python
audit_rows = [
    {"batch": "batch_1", "source": 100, "valid": 97, "rejected": 3},
    {"batch": "batch_2", "source": 120, "valid": 118, "rejected": 2},
    {"batch": "batch_3", "source": 90, "valid": 88, "rejected": 1},
]
```

Print whether each batch reconciles.

In [ ]:
# 'Print whether each batch reconciles.' It means do the source row counts match the processed row counts. 

audit_rows = [{"batch": "batch_1", "source": 100, "valid": 97, "rejected": 3}, {"batch": "batch_2", "source": 120, "valid": 118, "rejected": 2}, {"batch": "batch_3", "source": 90, "valid": 88, "rejected": 1}]

# So this is a list of dictionaries. 

reconciles = []
not_reconciles = []

for audit_row in audit_rows:
    if audit_row["source"] == audit_row["valid"] + audit_row["rejected"]:
        reconciles.append(audit_row["batch"]) # ' reconciles.append(audit_row)' appends the full dict, not just the batch.
    else: 
        not_reconciles.append(audit_row["batch"])

print(f"The following batches reconcile: {reconciles}. The following batches do not reconcile: {not_reconciles}.")
"""
To get rid of the square brackets and quote marks, use ", ".join(...), which glues the names together with a comma and space:
'print(f"The following batches reconcile: {', '.join(reconciles)}. The following batches do not reconcile: {', '.join(not_reconciles)}.")'.capitalize
"""

# Ironhacks Answer:
audit_rows = [{"batch": "batch_1", "source": 100, "valid": 97, "rejected": 3}, {"batch": "batch_2", "source": 120, "valid": 118, "rejected": 2}, {"batch": "batch_3", "source": 90, "valid": 88, "rejected": 1}]

for row in audit_rows:
    reconciles = (row["source"] == row["valid"] + row["rejected"])
 
print(row["batch"], "reconciles =", reconciles)

The following batches reconcile: ['batch_1', 'batch_2']. The following batches do not reconcile: ['batch_3'].


### Q. 16 — MCQ: Audit Record

Which audit value is most useful for checking whether records were lost during validation?

A. Only the city name

B. Only the file name

C. Source count, valid count, and rejected count

D. Only the Python version

========================================================================
ANSWER Q16
========================================================================

C. Source count, valid count, and rejected count .

========================================================================

## Optional Practice

### Q. 17 — Build a Complete Independent Daily Sales Pipeline

Use these three batches:

```python
import pandas as pd

batch_1 = pd.DataFrame({
    "order_id": [801, 802],
    "customer_id": [1, 2],
    "city": ["London", "Manchester"],
    "quantity": [2, 1],
    "unit_price_gbp": [25.00, 30.00],
    "status": ["complete", "complete"],
})

batch_2 = pd.DataFrame({
    "order_id": [803, 804],
    "customer_id": [1, 3],
    "city": [" London ", "Leeds"],
    "quantity": [1, 0],
    "unit_price_gbp": [40.00, 20.00],
    "status": ["complete", "complete"],
})

batch_3 = pd.DataFrame({
    "order_id": [805, 806],
    "customer_id": [4, 2],
    "city": ["Bristol", "Manchester"],
    "quantity": [3, 1],
    "unit_price_gbp": [None, 50.00],
    "status": ["complete", "cancelled"],
})
```

Create:

```python
raw_orders
valid_orders
rejected_orders
daily_sales
audit
```

Rules:

- preserve raw rows;
- reject quantity <= 0;
- reject missing price;
- trim city;
- calculate line_revenue_gbp;
- include only completed orders in daily sales;
- reconcile source, valid, and rejected counts.

In [58]:
batch_1 = pd.DataFrame({"order_id": [801, 802], "customer_id": [1, 2], "city": ["London", "Manchester"], "quantity": [2, 1], "unit_price_gbp": [25.00, 30.00], "status": ["complete", "complete"]})
batch_2 = pd.DataFrame({"order_id": [803, 804], "customer_id": [1, 3], "city": [" London ", "Leeds"], "quantity": [1, 0], "unit_price_gbp": [40.00, 20.00], "status": ["complete", "complete"]})
batch_3 = pd.DataFrame({"order_id": [805, 806], "customer_id": [4, 2], "city": ["Bristol", "Manchester"], "quantity": [3, 1], "unit_price_gbp": [None, 50.00], "status": ["complete", "cancelled"]})

batches = pd.concat([batch_1, batch_2, batch_3], ignore_index=True)
 
bronze = batches.copy()
 
invalid_mask = ((bronze["quantity"] <= 0) | bronze["unit_price_gbp"].isna())
rejected_orders = bronze[invalid_mask].copy()
valid_orders = bronze[~invalid_mask].copy()
 
valid_orders["city"] = (valid_orders["city"].str.strip())
valid_orders["line_revenue_gbp"] = (valid_orders["quantity"] * valid_orders["unit_price_gbp"])
silver = valid_orders[valid_orders["status"] == "complete"].copy()
 
gold = (silver.groupby("city", as_index=False).agg(completed_revenue_gbp=("line_revenue_gbp","sum")))

bronze_row_count = len(bronze)
silver_invalid_count = len(rejected_orders)
silver_valid_count = len(valid_orders)
bronze_to_silver_difference = bronze_row_count - (silver_invalid_count + silver_valid_count) 

audit = {"bronze_row_count": bronze_row_count, "silver_invalid_count": silver_invalid_count, "silver_valid_count": silver_valid_count, "bronze_to_silver_difference": bronze_to_silver_difference}
audit["reconciles"] = (audit["bronze_row_count"] == audit["silver_invalid_count"] + audit["silver_valid_count"]) # New code. 
print(audit)


{'bronze_row_count': 6, 'silver_invalid_count': 2, 'silver_valid_count': 4, 'bronze_to_silver_difference': 0, 'reconciles': True}


### Q. 18 — Detect a Dangerous Join

Use:

```python
import pandas as pd

customers = pd.DataFrame({
    "customer_id": [1, 1, 2],
    "customer_name": [
        "Amelia Clarke",
        "Amelia Clarke Duplicate",
        "Oliver Bennett",
    ],
})

orders = pd.DataFrame({
    "order_id": [901, 902],
    "customer_id": [1, 2],
    "amount_gbp": [45.50, 30.00],
})
```

Try a many-to-one join validation and explain why it should fail.

In [ ]:
customers = pd.DataFrame({"customer_id": [1, 1, 2], "customer_name": ["Amelia Clarke", "Amelia Clarke Duplicate", "Oliver Bennett"]})
 
orders = pd.DataFrame({"order_id": [901, 902], "customer_id": [1, 2], "amount_gbp": [45.50, 30.00]})

join = orders.merge(customers, on = "customer_id", how = "left", validate = "many_to_one")
print(join)

"""
Swapping customers and orders would run without an error, but that defeats the purpose.
We want the join to fail as a validation check, because customer_id should only appear
once in the customers table. Think of customer_id as its primary key, even though the
question doesn't say so.
"""

### Q. 19 — Write a Rejection Reason

Use:

```python
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003],
    "quantity": [0, 2, 3],
    "unit_price_gbp": [10.00, None, 20.00],
})
```

Create:

```python
rejection_reason
```

Values:

```python
invalid_quantity
missing_price
valid
```

In [3]:
orders = pd.DataFrame({"order_id": [1001, 1002, 1003], "quantity": [0, 2, 3], "unit_price_gbp": [10.00, None, 20.00]})

"""

The below is incorrect: 

invalid_quantity = []
missing_price = []
valid = []

for order in orders:
    if order[order["quantity"] < 1]:
        invalid_quantity.append(order["order_id"])
    elif order["unit_price_gbp"].isnull():
        missing_price.append(order["order_id"])
    else:
        valid.append(order["order_id"])

print(invalid_quantity)
print(missing_price)
print(valid)
"""

"""
# A
So the question to ask yourself is: “Am I looking at the whole table, or one row?”

Whole table: filter with table[table["col"] > x]
One row: check with if row["col"] > x:
"""

"""
# B
Looping straight over a DataFrame with `for order in orders:` gives you the column names (`"order_id"`, `"quantity"`, `"unit_price_gbp"`), 
not the rows, so you need `.iterrows()` to go through it row by row. But `.iterrows()` hands you two things together each time, the row number and the row itself, 
just like `.items()` gave you a key and a value with the dictionary. If you only give it one name, that name holds the whole pair and `order["quantity"]` won't work, 
so you need two names to split the pair: the first catches the row number and the second catches the row. You never actually use the row number anywhere in your code; 
it's only there so that `order` gets the row on its own, a bit like having to sign for both boxes in a two-item delivery even if you throw one away. That's why it's normally 
named `_`, which tells anyone reading your code "this slot needs a name, but I'm not using what's in it," giving you `for _, order in orders.iterrows():`.
"""

"""
# C
`.isnull()` (or `.isna()`) only works on pandas things like a whole table, row or column, never on a single plain value. In your orders loop, `.iterrows()` 
gives you one row at a time, so `order["unit_price_gbp"]` pulls out a single number, and adding `.isnull()` after it fails. Instead, you call it on the whole row first, 
`order.isnull()`, which gives a True/False for every column, then pick the one you want with `["unit_price_gbp"]`. In `bronze["unit_price_gbp"].isna()` there's no loop, 
so `bronze` is the whole table and picking the column still leaves you with a whole column, which does have `.isna()`. In short, on a whole table `.isna()` can go after 
the column, but inside a row loop `.isnull()` has to go on the row before you pick the column.
"""

invalid_quantity = []
missing_price = []
valid = []

for index, order in orders.iterrows(): #B
    if order["quantity"] < 1: # A
        invalid_quantity.append(int(order["order_id"]))
    elif order.isnull()["unit_price_gbp"]: # C
        missing_price.append(int(order["order_id"]))
    else:
        valid.append(int(order["order_id"]))

print(invalid_quantity)
print(missing_price)
print(valid)


# Ironhacks Answer:

# Appears to be new code. 
def reason(row):
    if row["quantity"] <= 0:
        return "invalid_quantity"
    if pd.isna(row["unit_price_gbp"]):
        return "missing_price"
    return "valid"
 
orders["rejection_reason"] = (orders.apply(reason, axis=1)) 
 
print(orders)

"""
This does the same job as your .iterrows() loop, but instead of putting order IDs into three separate lists, it adds a new 
column to the table saying why each row passed or failed.

`row["unit_price_gbp"]` is a single plain number, and a plain number has no `.isna()`, so adding it after would error. `pd.isna()` works because it's a 
standalone pandas function you hand a value to, and it accepts single numbers. You could also write `row.isna()["unit_price_gbp"]` (check the whole row, then pick the column), 
but `pd.isna(...)` is the more common way to check one value.

orders.apply(reason, axis=1) runs your reason function on each row of orders (axis=1 means row by row) and collects the answers into a new column. The outer brackets do nothing, 
so you can leave them out.
"""

[1001]
[1002]
[1003]
   order_id  quantity  unit_price_gbp  rejection_reason
0      1001         0            10.0  invalid_quantity
1      1002         2             NaN     missing_price
2      1003         3            20.0             valid


'\nThis does the same job as your .iterrows() loop, but instead of putting order IDs into three separate lists, it adds a new \ncolumn to the table saying why each row passed or failed.\n\n`row["unit_price_gbp"]` is a single plain number, and a plain number has no `.isna()`, so adding it after would error. `pd.isna()` works because it\'s a \nstandalone pandas function you hand a value to, and it accepts single numbers. You could also write `row.isna()["unit_price_gbp"]` (check the whole row, then pick the column), \nbut `pd.isna(...)` is the more common way to check one value.\n\norders.apply(reason, axis=1) runs your reason function on each row of orders (axis=1 means row by row) and collects the answers into a new column. The outer brackets do nothing, \nso you can leave them out.\n'

### Q. 20 — Design the Audit Fields

A team currently saves only:

```python
status = success
```

for every pipeline run.

Suggest a better audit record using only concepts covered today.

========================================================================
ANSWER Q20
========================================================================

'Saving only status = success proves nothing about the data. A run can “succeed” while quietly losing or duplicating rows. A better audit record would store a run_id so each run can be traced, plus source_count, valid_count and rejected_count, so you can check that source_count == valid_count + rejected_count. It should also hold a reconciles True/False flag from that check and simple data-quality counts such as duplicate order IDs, missing customer IDs and non-positive amounts. With those fields, status only says success when the counts reconcile and the checks pass, so it means something.'

========================================================================